In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys

sys.path.append("..")

from component.script import Project, Dataset, MWModel


## Set user parameters

In [ ]:
project_name = "mtq-refactor"

# Load the project from JSON
project = Project.load(project_name=project_name)


Loaded 1 model(s)
✓ Target set: forest_loss_2020_2024 (static)
✓ Features set: 9 variables
  Static: altitude, protected_area, rivers_dist, roads_dist, slope, subj
  Temporal (year: 2020): forest_gfc, forest_gfc_edge, towns_dist
✓ Target set: forest_loss_2015_2020 (static)
✓ Features set: 9 variables
  Static: altitude, protected_area, rivers_dist, roads_dist, slope, subj
  Temporal (year: 2015): forest_gfc, forest_gfc_edge, towns_dist
Loaded 2 dataset(s)
Project loaded from: /home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/mtq-refactor_project.json
Loaded 23 processed variables


In [ ]:
project.list_datasets()


['calibration', 'validation']

In [ ]:
calibration_dataset = project.get_dataset("calibration")
calibration_dataset


Dataset(target=LocalRasterVar(name='forest_loss_2015_2020', data_type='raster', year=None, active=True, tags=['deforestation', 'forest_loss', '2015_2020'], path=PosixPath('/home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/data/forest_loss_2015_2020_reprojected_matched.tif'), raster_type='categorical', post_processing=[], processing_history=['reprojected_matched'], default_crs='EPSG:5490', default_resolution=30.0), features=[LocalRasterVar(name='altitude', data_type='raster', year=None, active=True, tags=[], path=PosixPath('/home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/data/altitude_reprojected_matched.tif'), raster_type='continuous', post_processing=[], processing_history=['reprojected_matched'], default_crs='EPSG:5490', default_resolution=30.0), LocalRasterVar(name='forest_gfc', data_type='raster', year=2015, active=True, tags=['forest'], path=PosixPath('/home/jose/workspace/deforisk-nb-with-daniel/deforisk-ju

## Load MW datasets

MW datasets use the same binary raster convention as JNR.
The variables required per method are:

| Role | Variable | `fit()` | `apply()` |
|------|----------|---------|----------|
| Target | deforestation binary (0/1) tagged `"deforestation"` | ✓ | ✓ |
| Feature | `forest_edge` — distance-to-edge (m) at period start | ✓ | ✓ |
| Feature | `forest` — binary forest at period start | — | ✓ |

`fit()` only uses the target and `forest_edge_var`; the `forest_var` is only
needed by `apply()` for the `defrate_per_cat` computation.

## Instantiate the MW model

Override `forest_edge_var` / `forest_var` to match this project's variable names.
`defor_threshold` and `max_dist` can be set here as model-wide defaults
or overridden per individual `fit()` call.

`win_size_list` controls the moving window sizes in pixels.
One probability raster is produced per window size.

In [8]:
mw = MWModel(
    name="calibration_mw",
    forest_edge_var="forest_gfc_edge",  # matches the project variable name
    forest_var="forest_gfc",  # matches the project variable name
    win_size_list=[5, 11, 21],  # moving window sizes in pixels
    defor_threshold=99.5,  # distance percentile for the edge threshold
    max_dist=50000,  # maximum distance (m) for the distance-bin arange
    blk_rows=256,
)


## Fit — calibration period

Computes:
1. **`dist_thresh`** — distance-to-edge cutoff (m) from `rmj.dist_edge_threshold`
2. **`ldefrate_files`** — one local-deforestation-rate raster per window size
   from `rmj.local_defor_rate`

Only the deforestation target and the `forest_edge_var` feature are used in `fit()`.
The ldefrate rasters are reused for every `apply()` call (calibration, validation,
forecast) — no re-fitting needed per period.

In [9]:
mw.fit(
    dataset=calibration_dataset,
    time_interval=5,  # 2020 - 2015
    # defor_threshold=99.5,  # uncomment to override for this call only
    folder=project.folders.rmj_mw,
)

print(f"dist_thresh   : {mw.dist_thresh:.1f} m")
print(f"ldefrate_files: {list(mw.ldefrate_files.keys())} window sizes")



🔧 MW fit — period='calibration', windows=[5, 11, 21]
  dist_thresh=1080.0 m
  local_defor_rate — window 5×5 px...
  local_defor_rate — window 11×11 px...
  local_defor_rate — window 21×21 px...
✓ MW fit complete — 3 ldefrate files, trained_at=2026-04-01T16:32:55.654165
dist_thresh   : 1080.0 m
ldefrate_files: ['5', '11', '21'] window sizes


In [10]:
# Register with project — persists dist_thresh, ldefrate_files, and all metadata
# to the project JSON so the model survives a kernel restart.
mw.register(project)


  Model registered as project.models['mw_calibration_mw']
Project saved to: /home/jose/workspace/deforisk-nb-with-daniel/deforisk-jupyter-nb-v2/data/mtq-refactor/mtq-refactor_project.json


## Apply — calibration period

For each window size, produces:
1. A **probability map** GeoTIFF via `rmj.set_defor_cat_zero`
2. A **defrate CSV** (deforestation rate per category) via `rmj.defrate_per_cat`

`apply()` returns a `dict` mapping window size → probability raster `Path`.

In [11]:
cal_outputs = mw.apply(
    dataset=calibration_dataset,
    time_interval=5,  # 2020 - 2015
    output_folder=project.folders.rmj_mw,
)

print("Calibration probability maps:")
for win, path in cal_outputs.items():
    print(f"  window {win} → {path.name}")



🗺  MW apply — period='calibration', windows=[5, 11, 21]
  window 5 → prob_mw_5_calibration.tif
  window 11 → prob_mw_11_calibration.tif
  window 21 → prob_mw_21_calibration.tif
✓ MW apply complete — 3 probability maps written
Calibration probability maps:
  window 5 → prob_mw_5_calibration.tif
  window 11 → prob_mw_11_calibration.tif
  window 21 → prob_mw_21_calibration.tif


## Apply — validation period

The calibration `ldefrate_files` and `dist_thresh` are reused — the same model
instance is applied to the validation dataset without re-fitting.

In [12]:
validation_dataset = project.get_dataset("validation")

val_outputs = mw.apply(
    dataset=validation_dataset,
    time_interval=4,  # 2024 - 2020
    output_folder=project.folders.rmj_mw,
)

print("Validation probability maps:")
for win, path in val_outputs.items():
    print(f"  window {win} → {path.name}")



🗺  MW apply — period='validation', windows=[5, 11, 21]
  window 5 → prob_mw_5_validation.tif
  window 11 → prob_mw_11_validation.tif
  window 21 → prob_mw_21_validation.tif
✓ MW apply complete — 3 probability maps written
Validation probability maps:
  window 5 → prob_mw_5_validation.tif
  window 11 → prob_mw_11_validation.tif
  window 21 → prob_mw_21_validation.tif


## Reload after kernel restart

MW state (`dist_thresh`, `ldefrate_files`) lives in Pydantic fields —
no pickle file.  It reloads automatically from the project JSON.
`load_model()` verifies the ldefrate rasters are still on disk.

In [ ]:
project2 = Project.load(project_name="mtq-refactor")
mw_reloaded = project2.models["mw_calibration_mw"]
mw_reloaded.load_model()  # verifies ldefrate rasters exist on disk

print(f"dist_thresh   : {mw_reloaded.dist_thresh:.1f} m")
print(f"ldefrate_files: {list(mw_reloaded.ldefrate_files.keys())}")
